<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_6_Policy_Compliance_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [2]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [3]:
policy = {
    "confidential_data": "Confidential company information must not be shared with unauthorized external individuals.",
    "customer_data": "Customer personal information must not be shared publicly.",
    "file_sharing": "Company files may only be shared with authorized employees or approved business partners.",
    "approved_systems": "Confidential information must be handled using company-approved systems."
}

print("Company policy created successfully!")

Company policy created successfully!


In [4]:
synthetic_requests = [
    {
        "id": 1,
        "request": "Send a confidential financial report to an unauthorized person outside the company."
    },
    {
        "id": 2,
        "request": "Share a project document with an authorized employee using the company's approved system."
    },
    {
        "id": 3,
        "request": "Publish a customer's personal phone number on a public website."
    },
    {
        "id": 4,
        "request": "Send a company file to an approved business partner."
    },
    {
        "id": 5,
        "request": "Upload confidential company information to an unapproved external service."
    }
]

print("Synthetic policy requests created successfully!")

Synthetic policy requests created successfully!


In [5]:
def rule_based_evaluation(request):

    request_lower = request.lower()

    # Rule 1: Confidential data + unauthorized external sharing
    if (
        "confidential" in request_lower
        and (
            "unauthorized" in request_lower
            or "external" in request_lower
        )
    ):
        return "NON-COMPLIANT"

    # Rule 2: Customer personal information shared publicly
    if (
        ("customer" in request_lower or "personal" in request_lower)
        and (
            "public" in request_lower
            or "publicly" in request_lower
        )
    ):
        return "NON-COMPLIANT"

    # Rule 3: File shared with authorized employee
    if (
        "authorized employee" in request_lower
        and "approved" in request_lower
    ):
        return "COMPLIANT"

    # Rule 4: File shared with approved business partner
    if (
        "approved business partner" in request_lower
    ):
        return "COMPLIANT"

    # Rule 5: Confidential information sent to unapproved system
    if (
        "confidential" in request_lower
        and "unapproved" in request_lower
    ):
        return "NON-COMPLIANT"

    return "REQUIRES REVIEW"

In [6]:
for item in synthetic_requests:

    decision = rule_based_evaluation(item["request"])

    print("Request ID:", item["id"])
    print("Request:", item["request"])
    print("Decision:", decision)
    print("-" * 60)

Request ID: 1
Request: Send a confidential financial report to an unauthorized person outside the company.
Decision: NON-COMPLIANT
------------------------------------------------------------
Request ID: 2
Request: Share a project document with an authorized employee using the company's approved system.
Decision: COMPLIANT
------------------------------------------------------------
Request ID: 3
Request: Publish a customer's personal phone number on a public website.
Decision: NON-COMPLIANT
------------------------------------------------------------
Request ID: 4
Request: Send a company file to an approved business partner.
Decision: COMPLIANT
------------------------------------------------------------
Request ID: 5
Request: Upload confidential company information to an unapproved external service.
Decision: NON-COMPLIANT
------------------------------------------------------------


In [7]:
def compliance_explanation_agent(request, decision):

    prompt = f"""
You are a Policy Compliance Agent.

Company Policy:

1. Confidential company information must not be shared
   with unauthorized external individuals.

2. Customer personal information must not be shared publicly.

3. Company files may only be shared with authorized employees
   or approved business partners.

4. Confidential information must be handled using
   company-approved systems.

User Request:
{request}

Rule-Based Decision:
{decision}

Provide a short explanation for the decision.

If the decision is COMPLIANT, explain which policy condition
allows the request.

If the decision is NON-COMPLIANT, explain which policy rule
is violated.

If the decision is REQUIRES REVIEW, explain why additional
review is necessary.
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return interaction.output_text.strip()

In [8]:
def policy_compliance_agent(request):

    # Rule-based evaluation
    decision = rule_based_evaluation(request)

    # Gemini explanation
    explanation = compliance_explanation_agent(
        request,
        decision
    )

    print("========== POLICY COMPLIANCE AGENT ==========")

    print("\nUser Request:")
    print(request)

    print("\nRule-Based Decision:")
    print(decision)

    print("\nAI Explanation:")
    print(explanation)

In [9]:
policy_compliance_agent(
    "Send a confidential financial report to an unauthorized person outside the company."
)

========== POLICY COMPLIANCE AGENT ==========

User Request:
Send a confidential financial report to an unauthorized person outside the company.

Rule-Based Decision:
NON-COMPLIANT

AI Explanation:
**Explanation:**

This request is **NON-COMPLIANT** because it directly violates **Policy 1**, which strictly prohibits sharing confidential company information with unauthorized external individuals. 

It also violates **Policy 3**, which states that company files may only be shared with authorized employees or approved business partners.


In [10]:
print("========== SYNTHETIC POLICY TESTING ==========")

for item in synthetic_requests:

    decision = rule_based_evaluation(item["request"])

    print("\nRequest ID:", item["id"])
    print("Request:", item["request"])
    print("Decision:", decision)

========== SYNTHETIC POLICY TESTING ==========

Request ID: 1
Request: Send a confidential financial report to an unauthorized person outside the company.
Decision: NON-COMPLIANT

Request ID: 2
Request: Share a project document with an authorized employee using the company's approved system.
Decision: COMPLIANT

Request ID: 3
Request: Publish a customer's personal phone number on a public website.
Decision: NON-COMPLIANT

Request ID: 4
Request: Send a company file to an approved business partner.
Decision: COMPLIANT

Request ID: 5
Request: Upload confidential company information to an unapproved external service.
Decision: NON-COMPLIANT


In [11]:
print("""
========== POLICY COMPLIANCE WORKFLOW ==========

Synthetic User Request
          ↓
Rule-Based Evaluation
          ↓
COMPLIANT / NON-COMPLIANT / REQUIRES REVIEW
          ↓
Gemini Explanation Agent
          ↓
Final Compliance Explanation
""")


========== POLICY COMPLIANCE WORKFLOW ==========

Synthetic User Request
          ↓
Rule-Based Evaluation
          ↓
COMPLIANT / NON-COMPLIANT / REQUIRES REVIEW
          ↓
Gemini Explanation Agent
          ↓
Final Compliance Explanation

